# Notebook 4: Risk Analytics

**Project:** Automated Equity Research Platform

This notebook runs independently from the other notebooks.

## Standalone setup

This notebook is designed to run by itself. It can use a local clone, a Google Drive folder, or a GitHub repository.

Before publishing, replace `YOUR_USERNAME` in `REPO_URL` with your actual GitHub username.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

PROJECT_NAME = "Automated_Equity_Research_Platform"
REPO_URL = "https://github.com/YOUR_USERNAME/Automated_Equity_Research_Platform.git"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

candidate_roots = [
    Path.cwd(),
    Path("/content") / PROJECT_NAME,
    Path("/content/drive/MyDrive") / PROJECT_NAME,
    Path("/content/drive/MyDrive/AI_Equity_Research_Platform"),
]

PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "pyproject.toml").exists()),
    None,
)

if PROJECT_ROOT is None:
    if "YOUR_USERNAME" in REPO_URL:
        raise RuntimeError(
            "Project files were not found. Upload the full project folder to Google Drive "
            "or replace YOUR_USERNAME in REPO_URL with your real GitHub username."
        )
    target = Path("/content") / PROJECT_NAME
    if not target.exists():
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
    PROJECT_ROOT = target

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .

In [ ]:
import os

try:
    from google.colab import userdata
    for secret_name in ("SEC_USER_AGENT", "FRED_API_KEY"):
        try:
            value = userdata.get(secret_name)
            if value:
                os.environ[secret_name] = value
        except Exception:
            pass
except Exception:
    pass

print("SEC enabled:", bool(os.getenv("SEC_USER_AGENT")))
print("FRED enabled:", bool(os.getenv("FRED_API_KEY")))

Calculate risk metrics for one stock and portfolio optimization when peers are provided.

In [ ]:
from equity_research.data import YahooFinanceClient
from equity_research.risk import RiskAnalyzer, PortfolioOptimizer
from equity_research.visualization import (
    drawdown_chart, correlation_heatmap, portfolio_weights_chart, efficient_frontier_chart
)
import pandas as pd
import plotly.graph_objects as go

TICKER = input("Main ticker: ").strip().upper() or "AAPL"
peer_input = input("Peers separated by commas, or blank: ").strip()
PEERS = [p.strip().upper() for p in peer_input.split(",") if p.strip()]
TICKERS = list(dict.fromkeys([TICKER] + [p for p in PEERS if p != TICKER]))
BENCHMARK = input("Benchmark, default SPY: ").strip().upper() or "SPY"

yahoo = YahooFinanceClient()
close = yahoo.close_matrix(TICKERS + [BENCHMARK], start="2018-01-01", auto_adjust=True)
display(close.tail())

In [ ]:
risk = RiskAnalyzer(annualization=252, risk_free_rate=0.04)

metrics = risk.performance_metrics(close[TICKER], benchmark_prices=close[BENCHMARK])
display(pd.Series(metrics, name="Value").to_frame())

drawdown_chart(risk.drawdown_series(close[TICKER]), TICKER).show()

In [ ]:
comparison_columns = [t for t in TICKERS + [BENCHMARK] if t in close.columns]
normalized = close[comparison_columns].divide(close[comparison_columns].apply(lambda c: c.dropna().iloc[0])).multiply(100)

fig = go.Figure()
for t in normalized.columns:
    fig.add_trace(go.Scatter(x=normalized.index, y=normalized[t], mode="lines", name=t))
fig.update_layout(title="Growth of $100", template="plotly_white", height=550)
fig.show()

In [ ]:
available_tickers = [t for t in TICKERS if t in close.columns]
if len(available_tickers) >= 2:
    asset_prices = close[available_tickers].dropna()
    returns = asset_prices.pct_change(fill_method=None).dropna()
    correlation_heatmap(returns).show()

    optimizer = PortfolioOptimizer(annualization=252, risk_free_rate=0.04)
    max_weight = max(0.40, 1 / len(available_tickers))

    max_sharpe = optimizer.maximize_sharpe(asset_prices, max_weight=max_weight)
    min_variance = optimizer.minimum_variance(asset_prices, max_weight=max_weight)

    print("Maximum-Sharpe Portfolio")
    display(max_sharpe.frame())
    portfolio_weights_chart(max_sharpe.weights, "Maximum-Sharpe Portfolio").show()

    print("Minimum-Variance Portfolio")
    display(min_variance.frame())
    portfolio_weights_chart(min_variance.weights, "Minimum-Variance Portfolio").show()

    frontier = optimizer.efficient_frontier(asset_prices, points=35, max_weight=max_weight)
    if not frontier.empty:
        efficient_frontier_chart(frontier).show()
else:
    print("Portfolio optimization skipped. Enter at least one peer.")